In [ ]:
!pip install ultralytics openai
!pip install ultralytics
!pip install google-generativeai -q

In [353]:
# -*- coding: utf-8 -*-
"""
成员C：多模态安全巡检系统
"""

import json
import os
from typing import Dict, List
from ultralytics import YOLO
from openai import OpenAI

CLASS_PERSON = "person"
CLASS_HELMET = "helmet"
CLASS_NO_HELMET = "no-helmet"

class SafetyDetectorAPI:  # （成员B交付）
    def __init__(self, model_path: str):
        self.is_mock = not os.path.exists(model_path)
        if not self.is_mock:
            self.model = YOLO(model_path, verbose=False)

    def analyze_image(self, image_path: str, force_scene: str = None) -> str:
        if self.is_mock or not os.path.exists(image_path):
            # 模拟数据（略）
            return json.dumps({"details": []})
        results = self.model(image_path, verbose=False, conf=0.35)  # 适当提高阈值减少误检
        detections = []
        for box in results[0].boxes:
            class_id = int(box.cls[0].item())
            detections.append({
                "object": self.model.names[class_id],
                "bounding_box": box.xyxy[0].tolist()
            })
        return json.dumps({"details": detections})

class SafetyLLMReporter:   # （成员C核心）
    def __init__(self, api_key: str, model: str = "deepseek-chat", use_zero_shot: bool = False):
        self.client = OpenAI(api_key=api_key, base_url="https://api.deepseek.com")
        self.model = model
        self.use_zero_shot = use_zero_shot

    def _filter_handheld(self, details: List[Dict]) -> List[Dict]:
        # ... 手持误检过滤逻辑保持不变 ...
        persons = [d for d in details if d.get("object") == CLASS_PERSON]
        helmets = [d for d in details if d.get("object") == CLASS_HELMET]
        no_helmets = [d for d in details if d.get("object") == CLASS_NO_HELMET]
        kept_no_helmets = []
        for nh in no_helmets:
            nh_box = nh["bounding_box"]
            near_helmet = any((( (nh_box[0]+nh_box[2])/2 - (h["bounding_box"][0]+h["bounding_box"][2])/2 )**2 +
                               ((nh_box[1]+nh_box[3])/2 - (h["bounding_box"][1]+h["bounding_box"][3])/2 )**2 )**0.5 < 60
                              for h in helmets)
            nh_center_y = (nh_box[1] + nh_box[3]) / 2
            nh_center_x = (nh_box[0] + nh_box[2]) / 2
            on_head = False
            for p in persons:
                p_box = p["bounding_box"]
                if (p_box[0] < nh_center_x < p_box[2] and
                    p_box[1] < nh_center_y < p_box[1] + (p_box[3] - p_box[1]) / 3):
                    on_head = True
                    break
            if near_helmet and not on_head:
                continue
            kept_no_helmets.append(nh)
        return [d for d in details if d.get("object") != CLASS_NO_HELMET] + kept_no_helmets

    def _parse_data(self, json_str: str) -> Dict:
        try:
            data = json.loads(json_str)
        except:
            data = {}
        details = data.get("details", [])
        details = self._filter_handheld(details)
        p_count = sum(1 for d in details if d.get("object") == CLASS_PERSON)
        n_count = sum(1 for d in details if d.get("object") == CLASS_NO_HELMET)
        h_count = sum(1 for d in details if d.get("object") == CLASS_HELMET)

        # ✅ 增强型场景判定（容忍少量误检person）
        if not details:
            scene = "disturb"
        elif n_count > 0:
            scene = "violation"
        elif p_count == 0 and h_count > 0:
            scene = "helmet_only"
        elif 0 < p_count <= 2 and h_count >= p_count * 2 and n_count == 0:
            scene = "helmet_only"
        else:
            scene = "safe"

        return {
            "scene": scene, "p_count": p_count,
            "n_count": n_count, "h_count": h_count,
            "raw": json.dumps({"details": details})
        }

    def process_and_report(self, json_str: str) -> str:
        info = self._parse_data(json_str)
        if info["scene"] == "disturb":
            broadcast = "🔊 巡检播报：当前监控未检测到有效人员目标，现场状况正常。"
        elif info["scene"] == "helmet_only":
            broadcast = "🔊 巡检播报：检测到安全帽但未发现人员，请人工复核现场。"
        elif info["n_count"] == 0:
            broadcast = "🔊 巡检播报：全员安全装备佩戴齐全，状况正常。"
        else:
            broadcast = f"🔊 巡检播报：发现 {info['n_count']} 人未佩戴安全帽，请立即纠正！"
        return broadcast

    def answer_question(self, question: str, json_str: str) -> str:
        info = self._parse_data(json_str)
        q_lower = question.lower()
        if info["scene"] == "helmet_only":
            if "没戴安全帽" in question or "未戴安全帽" in question:
                return "🤖 答：画面中检测到安全帽，但未检测到人员，无法判断是否有人未戴安全帽。"
            if "几个人" in question or "人数" in question:
                return "🤖 答：画面中未检测到人员。"
            if "安全帽" in question and ("佩戴" in question or "齐全" in question):
                return "🤖 答：检测到安全帽存在，但未检测到人员，建议人工确认佩戴情况。"
        # 其他问答分支（略，保持原样）
        if "没戴安全帽" in question or "未戴安全帽" in question:
            return f"🤖 答：根据检测数据，共有 {info['n_count']} 人未佩戴安全帽。"
        if "几个人" in question or "人数" in question:
            return f"🤖 答：画面中检测到 {info['p_count']} 人。"
        if "安全帽" in question and ("佩戴" in question or "齐全" in question):
            if info["n_count"] == 0:
                return "🤖 答：是的，所有检测到的人员均佩戴了安全帽。"
            else:
                return f"🤖 答：否，有 {info['n_count']} 人未佩戴安全帽。"
        # 默认回答
        return "🤖 答：请提出具体问题。"

# 测试
if __name__ == "__main__":
    API_KEY = "sk-ddd77aa3129c4f4ca8f0aafb5675b634"   # 密钥

    detector = SafetyDetectorAPI("best.pt")
    reporter = SafetyLLMReporter(API_KEY, use_zero_shot=True)  # 可改为True/False来切换零样本/少样本

    print("=" * 60)
    print("成员C 多模态安全巡检系统")
    print(f"当前模式：{'零样本 (Zero-shot)' if reporter.use_zero_shot else '少样本 (Few-shot)'}")
    print("=" * 60)

    v_json = detector.analyze_image("3_3.jpg", force_scene=None)   # 选择测试图片
    print("📷 视觉数据已准备。")
    print(reporter.process_and_report(v_json))
    print("👤 问：有几个人没戴安全帽？")
    print(reporter.answer_question("有几个人没戴安全帽？", v_json))
    print("-" * 50)

成员C 多模态安全巡检系统
当前模式：零样本 (Zero-shot)
📷 视觉数据已准备。
🔊 巡检播报：检测到安全帽但未发现人员，请人工复核现场。
👤 问：有几个人没戴安全帽？
🤖 答：画面中检测到安全帽，但未检测到人员，无法判断是否有人未戴安全帽。
--------------------------------------------------
